In [1]:
!pip install openai

In [2]:
import openai
import os

In [3]:
import os
import re

def read_file_clean(location):
    """
    Reads a C source code file, removes comments, newlines, and tabs,
    and returns compact code suitable for functional comparison.

    Parameters:
        location (str): Path to the C source code file.

    Returns:
        str: Cleaned and compact code.

    Raises:
        FileNotFoundError: If the file does not exist.
        PermissionError: If reading is not allowed.
        IOError: For other I/O issues.
    """
    if not os.path.isfile(location):
        raise FileNotFoundError(f"The file at '{location}' does not exist.")

    try:
        with open(location, 'r', encoding='utf-8') as file:
            code = file.read()

        # Remove block comments (/* ... */)
        code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)

        # Remove line comments (// ...)
        code = re.sub(r'//.*', '', code)

        # Remove tabs, newlines, and multiple spaces
        code = code.replace('\n', ' ').replace('\t', ' ')
        code = re.sub(r'\s+', ' ', code).strip()

        return code

    except PermissionError:
        raise PermissionError(f"Permission denied while reading '{location}'.")
    except IOError as e:
        raise IOError(f"An I/O error occurred while reading '{location}': {e}")



In [4]:
def get_system_prompt():
    return 'You are a helpful and precise coding assistant. Your job is to identify whether two code snippets behave differently in terms of functionality.'



In [5]:
def get_cot_system_prompt():
    return (
        "You are an expert in program analysis and formal verification.\n\n"
        "Your task is to determine semantic (observational) equivalence between a source program and a mutant program.\n\n"
        "You must reason about program behavior, not just syntax.\n\n"
        "Definition:\n"
        "Two programs are observationally equivalent if, for every feasible input, their observable outputs are identical.\n\n"
        "Critical requirements:\n"
        "- Do NOT conclude non-equivalence based only on syntactic differences.\n"
        "- You MUST analyze how differences propagate through the program.\n"
        "- You MUST check whether differences are masked before reaching the output.\n"
        "- You MUST consider interaction between multiple bugs.\n\n"
        "You must explicitly reason about the following masking effects:\n"
        "1. Overwrite masking (later assignments remove divergence)\n"
        "2. Control-flow masking (divergent code not executed)\n"
        "3. Output masking (internal difference does not affect output)\n"
        "4. Interaction masking (one bug cancels or neutralizes another)\n"
        "5. Path-specific masking (difference appears only on some paths)\n\n"
        "For each identified difference:\n"
        "- Identify affected variables or predicates\n"
        "- Trace forward propagation\n"
        "- Determine whether divergence survives or is masked\n\n"
        "For multi-bug mutants:\n"
        "- Analyze combined effects, not just each bug independently\n"
        "- Check if bugs amplify, cancel, or hide each other\n\n"
        "Do NOT skip propagation analysis.\n\n"
        "Final decision rules:\n"
        "- Equivalent -> no feasible input causes output difference\n"
        "- Non-equivalent -> at least one feasible input causes output difference\n"
        "- Conditionally non-equivalent -> only some paths diverge\n"
        "- Masked divergence -> internal differences exist but do not reach output\n\n"
        "Be precise, structured, and conservative in conclusions.\n"
        "If uncertain, explicitly state limitations."
    )

In [6]:
def get_user_prompt(source,mutant):
    prompt = f"Here is the original (source) code:\n\n{source}\n\nAnd here is the mutated version of the code:\n\n{mutant}\n\nDo these two pieces of code have any **functional difference**? Respond strictly with \"Yes\" or \"No\" only. Explain your answer in single line if required provide code responsible with line number in mutant, if there are multiple discrepancies each will have one line explanation."
    return prompt

In [7]:
def get_cot_user_prompt(source, mutant):
    prompt = (
        "Task:\n"
        "Compare the following source program (S) and mutant program (M) for observational equivalence.\n\n"
        "Follow the required reasoning steps carefully.\n\n"
        "Steps to perform:\n\n"
        "1. Identify all semantic differences between S and M.\n"
        "2. For each difference:\n"
        " - List directly affected variables or conditions\n"
        " - Describe immediate effect\n"
        "3. Trace forward propagation:\n"
        " - How does this difference influence later computations?\n"
        " - Does it affect control flow?\n"
        " - Does it reach output variables?\n"
        "4. Perform masking analysis:\n"
        " - Is the difference overwritten later?\n"
        " - Is it blocked by path conditions?\n"
        " - Is it masked at output?\n"
        "5. Perform interaction analysis:\n"
        " - Do multiple differences interact?\n"
        " - Does one difference cancel or amplify another?\n"
        " - Check explicitly for one-way masking (one bug/change masks or hides the effect of another).\n"
        " - Check explicitly for two-way masking (two or more bugs/changes cancel each other so that combined behavior appears correct).\n"
        " - IMPORTANT: If any such masking occurs, you MUST say so explicitly, and identify which bugs/changes are involved.\n"
        "6. Perform path-sensitive reasoning if needed.\n"
        "7. Determine whether any feasible input leads to different outputs.\n\n"
        "Output requirements (STRICT):\n"
        "- Return ONLY valid JSON. No markdown, no code fences, no extra text.\n"
        "- The JSON must include an upfront answer that starts with Yes/No and a one-line justification.\n"
        "- That Yes/No and one-liner MUST explicitly state whether any bug/change is masked by another bug/change (e.g., Yes, outputs differ and no masking between bugs; or No, outputs are equivalent because bug B masks bug A).\n"
        "- The JSON must use these exact top-level keys: answer, one_liner, A_observable_output_variables, B_differences_identified, C_propagation_analysis, D_masking_analysis, E_interaction_analysis, F_surviving_divergences_reaching_output, G_final_verdict, H_witness_input_or_condition, I_confidence_and_limitations.\n"
        "- The meaning of the keys is as follows:\n"
        " - answer: Yes/No to \"Are S and M observationally equivalent?\"\n"
        " - one_liner: one sentence, starting with Yes/No, that explicitly mentions whether masking between bugs is present or absent.\n"
        " - A_observable_output_variables: list of variables considered as observable outputs.\n"
        " - B_differences_identified: list of semantic differences (with ids, descriptions, directly_affected, immediate_effect).\n"
        " - C_propagation_analysis: mapping from difference id to how it propagates.\n"
        " - D_masking_analysis: structured description of overwrite, control_flow, output, interaction, path_specific masking.\n"
        " - E_interaction_analysis: summary of interactions between multiple differences.\n"
        " - F_surviving_divergences_reaching_output: list of differences that still reach outputs (with how).\n"
        " - G_final_verdict: one of Equivalent, Non-equivalent, Conditionally non-equivalent, Masked divergence.\n"
        " - H_witness_input_or_condition: concrete or symbolic input/condition if non-equivalent.\n"
        " - I_confidence_and_limitations: object with numeric confidence and textual limitations.\n\n"
        "---\n\n"
        "Source Program (S):\n"
        f"{source}\n\n"
        "---\n\n"
        "Mutant Program (M):\n"
        f"{mutant}"
    )
    return prompt

In [ ]:
from openai import OpenAI
# Create OpenAI client instance
client = OpenAI(api_key='openai.api_key')


In [9]:
# Updated function for a single message input
def send_prompt(message, model="gpt-4o", temperature=0):
    """
    Send a single user prompt to the OpenAI Chat API with a system message.
    
    Parameters:
        message (str): The user's prompt.
        model (str): The OpenAI model to use (e.g., 'gpt-4o').
        temperature (float): Sampling temperature for response randomness.

    Returns:
        reply (str): The assistant's response.
        usage (dict): Token usage details.
    """
    messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": message}
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        reply = response.choices[0].message.content
        usage = response.usage
        return reply, usage
    except Exception as e:
        print(model)
        print("OpenAI API error:", str(e))
        return None, None


def send_cot_prompt(message, model="gpt-5.2", temperature=0):
    """
    Send a single user prompt to the OpenAI Chat API with the CoT system message.

    Parameters:
        message (str): The user's prompt.
        model (str): The OpenAI model to use (default: 'gpt-5.2').
        temperature (float): Sampling temperature for response randomness.

    Returns:
        reply (str): The assistant's response.
        usage (dict): Token usage details.
    """
    messages = [
        {"role": "system", "content": get_cot_system_prompt()},
        {"role": "user", "content": message}
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        reply = response.choices[0].message.content
        usage = response.usage
        return reply, usage
    except Exception as e:
        print(model)
        print("OpenAI API error:", str(e))
        return None, None

In [10]:
explaination,usage = send_prompt(get_user_prompt(read_file_clean('./source/tcas_source.c'),read_file_clean('./source/tcas_v10_source.c')))

In [11]:
explaination

'Yes. The mutated code changes the conditions in `Own_Below_Threat()` and `Own_Above_Threat()` functions by using `<=` instead of `<` (lines 66 and 67 in the mutated version), which affects the logic of threat assessment.'

In [12]:
usage

CompletionUsage(completion_tokens=54, prompt_tokens=2309, total_tokens=2363, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=2304))

In [13]:
explaination_11,usage_11 = send_prompt(get_user_prompt(read_file_clean('./source/tcas_source.c'),read_file_clean('./source/tcas_v11_source.c')))

In [14]:
explaination_15,usage_15 = send_prompt(get_user_prompt(read_file_clean('./source/tcas_source.c'),read_file_clean('./source/tcas_v15_source.c')))

In [15]:
explaination_31,usage_31 = send_prompt(get_user_prompt(read_file_clean('./source/tcas_source.c'),read_file_clean('./source/tcas_v31_source.c')))

In [16]:
explaination_32,usage_32 = send_prompt(get_user_prompt(read_file_clean('./source/tcas_source.c'),read_file_clean('./source/tcas_v32_source.c')))

In [17]:
explaination_40,usage_40 = send_prompt(get_user_prompt(read_file_clean('./source/tcas_source.c'),read_file_clean('./source/tcas_v40_source.c')))

In [18]:
print(explaination)
print()
print(explaination_11)
print()
print(explaination_15)
print()
print(explaination_31)
print()
print(explaination_32)
print()
print(explaination_40)

Yes. The mutated code changes the conditions in `Own_Below_Threat()` and `Own_Above_Threat()` functions by using `<=` instead of `<` (lines 66 and 67 in the mutated version), which affects the logic of threat assessment.

Yes. The conditions in `Own_Below_Threat()` and `Own_Above_Threat()` have been changed from `<` to `<=` in the mutated version, which affects the logic of threat assessment.

Yes. 

1. The mutated code changes `#define MINSEP 300` to `#define MINSEP 300+350`, affecting separation logic (line 3).
2. The condition `Cur_Vertical_Sep > MAXALTDIFF` is removed from `enabled` in `alt_sep_test()` (line 79).

Yes. The mutated code adds additional conditions in the `Non_Crossing_Biased_Climb` function (lines with `result = result && (Own_Tracked_Alt <= Other_Tracked_Alt);` and `result = result && (Own_Tracked_Alt < Other_Tracked_Alt);`), which can change the behavior of the function compared to the original code.

Yes. The mutated code adds additional conditions in `Non_Crossin

In [19]:
explaination_cot_10, usage_cot_10 = send_cot_prompt(
    get_cot_user_prompt(
        read_file_clean('./source/tcas_source.c'),
        read_file_clean('./source/tcas_v10_source.c')
    )
)

In [20]:
explaination_cot_11, usage_cot_11 = send_cot_prompt(
    get_cot_user_prompt(
        read_file_clean('./source/tcas_source.c'),
        read_file_clean('./source/tcas_v11_source.c')
    )
)

In [21]:
explaination_cot_15, usage_cot_15 = send_cot_prompt(
    get_cot_user_prompt(
        read_file_clean('./source/tcas_source.c'),
        read_file_clean('./source/tcas_v15_source.c')
    )
)

In [22]:
explaination_cot_31, usage_cot_31 = send_cot_prompt(
    get_cot_user_prompt(
        read_file_clean('./source/tcas_source.c'),
        read_file_clean('./source/tcas_v31_source.c')
    )
)

In [23]:
explaination_cot_32, usage_cot_32 = send_cot_prompt(
    get_cot_user_prompt(
        read_file_clean('./source/tcas_source.c'),
        read_file_clean('./source/tcas_v32_source.c')
    )
)

In [24]:
explaination_cot_40, usage_cot_40 = send_cot_prompt(
    get_cot_user_prompt(
        read_file_clean('./source/tcas_source.c'),
        read_file_clean('./source/tcas_v40_source.c')
    )
)

In [25]:
print(explaination_cot_10)
print(usage_cot_10)

{
  "answer": "No",
  "one_liner": "No, S and M are not observationally equivalent; outputs differ for feasible inputs and there is no masking between the two changes (they reinforce divergence when altitudes are equal).",
  "A_observable_output_variables": [
    "stdout printed integer from alt_sep_test()",
    "process exit code (0 on normal path, 1 on argc<13 error path)"
  ],
  "B_differences_identified": [
    {
      "id": "D1",
      "description": "Own_Below_Threat uses strict < in S but non-strict <= in M.",
      "directly_affected": [
        "predicate Own_Below_Threat()",
        "all callers of Own_Below_Threat (Non_Crossing_Biased_Climb, Non_Crossing_Biased_Descend, alt_sep_test)"
      ],
      "immediate_effect": "When Own_Tracked_Alt == Other_Tracked_Alt, S returns false while M returns true; otherwise behavior matches."
    },
    {
      "id": "D2",
      "description": "Own_Above_Threat uses strict < in S but non-strict <= in M.",
      "directly_affected": [
     

In [26]:
print(explaination_cot_11)
print(usage_cot_11)

{
  "answer": "No",
  "one_liner": "No, outputs can differ and there is no masking between bugs; the <= changes plus removal of the (need_upward_RA && need_downward_RA) UNRESOLVED case allow a different RA on equal-altitude inputs.",
  "A_observable_output_variables": [
    "stdout printed integer from alt_sep_test()",
    "process exit code (0 vs 1 on argc check)"
  ],
  "B_differences_identified": [
    {
      "id": "D1",
      "description": "Own_Below_Threat uses < in S but <= in M.",
      "directly_affected": [
        "predicate Own_Below_Threat()"
      ],
      "immediate_effect": "When Own_Tracked_Alt == Other_Tracked_Alt, S returns false while M returns true; otherwise same."
    },
    {
      "id": "D2",
      "description": "Own_Above_Threat uses < in S but <= in M.",
      "directly_affected": [
        "predicate Own_Above_Threat()"
      ],
      "immediate_effect": "When Own_Tracked_Alt == Other_Tracked_Alt, S returns false while M returns true; otherwise same."
    

In [27]:
print(explaination_cot_15)
print(usage_cot_15)

{
  "answer": "No",
  "one_liner": "No, S and M are not observationally equivalent; outputs differ for feasible inputs and there is no masking between the two changes (each can independently change the printed alt_sep_test result).",
  "A_observable_output_variables": [
    "stdout printed integer from alt_sep_test()",
    "process exit code (0 on normal path, 1 on argc error path)"
  ],
  "B_differences_identified": [
    {
      "id": "D1",
      "description": "Constant MINSEP changed from 300 (S) to 300+350=650 (M).",
      "directly_affected": [
        "MINSEP",
        "predicates (Cur_Vertical_Sep >= MINSEP) inside Non_Crossing_Biased_Climb/Descend else/then branches"
      ],
      "immediate_effect": "Tightens the Cur_Vertical_Sep threshold used in some RA feasibility checks; some cases that pass in S (>=300) fail in M unless Cur_Vertical_Sep>=650."
    },
    {
      "id": "D2",
      "description": "enabled condition in alt_sep_test changed: S requires (Cur_Vertical_Sep > M

In [28]:
print(explaination_cot_31)
print(usage_cot_31)

{
  "answer": "No",
  "one_liner": "No, S and M are not observationally equivalent; outputs differ for feasible inputs and there is no masking between the changes (they reinforce divergence rather than cancel it).",
  "A_observable_output_variables": [
    "stdout printed integer alt_sep_test() return value (0/1/2)",
    "process exit code (always 0 on normal path, 1 on argc<13 path)"
  ],
  "B_differences_identified": [
    {
      "id": "D1",
      "description": "In alt_sep_test, S computes need_upward_RA = Non_Crossing_Biased_Climb() && Own_Below_Threat(); M computes need_upward_RA = Non_Crossing_Biased_Climb() (drops the explicit Own_Below_Threat() conjunct).",
      "directly_affected": [
        "need_upward_RA",
        "alt_sep (via subsequent if/else)",
        "control-flow of RA selection"
      ],
      "immediate_effect": "M may set need_upward_RA true in cases where Own_Below_Threat() is false, provided Non_Crossing_Biased_Climb() returns true."
    },
    {
      "id": 

In [29]:
print(explaination_cot_32)
print(usage_cot_32)

{
  "answer": "No",
  "one_liner": "No, S and M are not observationally equivalent; outputs differ for feasible inputs and there is no masking between changes (the added conjuncts in Non_Crossing_Biased_Descend and the removed Own_Above_Threat gate in alt_sep_test both independently change when DOWNWARD_RA is issued).",
  "A_observable_output_variables": [
    "stdout printed value of alt_sep_test() (0/1/2)"
  ],
  "B_differences_identified": [
    {
      "id": "D1",
      "description": "In alt_sep_test, mutant computes need_downward_RA = Non_Crossing_Biased_Descend() instead of Non_Crossing_Biased_Descend() && Own_Above_Threat().",
      "directly_affected": [
        "need_downward_RA",
        "predicate for selecting DOWNWARD_RA"
      ],
      "immediate_effect": "M may set need_downward_RA true even when Own_Above_Threat() is false (i.e., when Own_Tracked_Alt <= Other_Tracked_Alt)."
    },
    {
      "id": "D2",
      "description": "In Non_Crossing_Biased_Descend (upward_pref

In [30]:
print(explaination_cot_40)
print(usage_cot_40)

{
  "answer": "No",
  "one_liner": "No, outputs can differ and there is no masking between the changes; each change can independently alter the final advisory.",
  "A_observable_output_variables": [
    "stdout printed integer from alt_sep_test() (UNRESOLVED=0, UPWARD_RA=1, DOWNWARD_RA=2)",
    "process exit code (0 on normal path, 1 on argc error path)"
  ],
  "B_differences_identified": [
    {
      "id": "D1",
      "description": "Non_Crossing_Biased_Climb() upward_preferred branch weakened: S returns !(Own_Below_Threat()) || (Own_Below_Threat() && !(Down_Separation>=ALIM())); M returns (Own_Below_Threat() && !(Down_Separation>=ALIM())) only.",
      "directly_affected": [
        "result in Non_Crossing_Biased_Climb (upward_preferred branch)",
        "return value of Non_Crossing_Biased_Climb()"
      ],
      "immediate_effect": "When upward_preferred is true and Own_Below_Threat() is false, S returns true but M returns false."
    },
    {
      "id": "D2",
      "description"